In [0]:

# Load the data

from pyspark.sql import functions as F

df = spark.table("CCC")

display(df.limit(10))

#Avg balance
bill_cols = [
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"
]

df = df.withColumn(
    "AVG_BILL_AMT",
    sum(F.col(c) for c in bill_cols) / 6
)

#avg payment
payment_cols = [
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

df = df.withColumn(
    "AVG_PAY_AMT",
    sum(F.col(c) for c in payment_cols) / 6
)


#Utilization
df = df.withColumn(
    "AVG_UTILIZATION",
    F.when(
        F.col("LIMIT_BAL") > 0,
        F.col("AVG_BILL_AMT") / F.col("LIMIT_BAL")
    )
)


#payment to balance
df = df.withColumn(
    "PAYMENT_TO_BALANCE",
    F.when(
        F.col("AVG_BILL_AMT") > 0,
        F.col("AVG_PAY_AMT") / F.col("AVG_BILL_AMT")
    )
)


#delinquent
pay_status_cols = [
    "PAY_1", "PAY_2", "PAY_3",
    "PAY_4", "PAY_5", "PAY_6"
]

df = df.withColumn(
    "DELINQUENT_MONTHS",
    sum(
        F.when(F.col(c) > 0, 1).otherwise(0)
        for c in pay_status_cols
    )
)

#max delinquent
df = df.withColumn(
    "MAX_DELINQUENCY",
    F.greatest(*[F.col(c) for c in pay_status_cols])
)

#balance trend
df = df.withColumn(
    "BALANCE_CHANGE",
    F.col("BILL_AMT1") - F.col("BILL_AMT6")
)


#sanity check
display(
    df.select(
        "ID",
        "LIMIT_BAL",
        "AVG_BILL_AMT",
        "AVG_PAY_AMT",
        "AVG_UTILIZATION",
        "PAYMENT_TO_BALANCE",
        "DELINQUENT_MONTHS",
        "MAX_DELINQUENCY",
        "BALANCE_CHANGE",
        "dpnm"
    ).limit(20)
)

#save dataset
df.write.mode("overwrite").saveAsTable("CCC_FEATURES")

ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,dpnm
1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0
6,50000,1,1,2,37,0,0,0,0,0,0,64400,57069,57608,19394,19619,20024,2500,1815,657,1000,1000,800,0
7,500000,1,1,2,29,0,0,0,0,0,0,367965,412023,445007,542653,483003,473944,55000,40000,38000,20239,13750,13770,0
8,100000,2,2,2,23,0,-1,-1,0,0,-1,11876,380,601,221,-159,567,380,601,0,581,1687,1542,0
9,140000,2,3,1,28,0,0,2,0,0,0,11285,14096,12108,12211,11793,3719,3329,0,432,1000,1000,1000,0
10,20000,1,3,2,35,-2,-2,-2,-2,-1,-1,0,0,0,0,13007,13912,0,0,0,13007,1122,0,0


ID,LIMIT_BAL,AVG_BILL_AMT,AVG_PAY_AMT,AVG_UTILIZATION,PAYMENT_TO_BALANCE,DELINQUENT_MONTHS,MAX_DELINQUENCY,BALANCE_CHANGE,dpnm
1,20000,1284.0,114.83333333333333,0.0642,0.08943406022845275,2,2,3913,1
2,120000,2846.1666666666665,833.3333333333334,0.023718055555555555,0.2927914739122797,2,2,-579,1
3,90000,16942.166666666668,1836.3333333333333,0.1882462962962963,0.10838834072776996,0,0,13690,0
4,50000,38555.666666666664,1398.0,0.7711133333333333,0.03625926150068732,0,0,17443,0
5,50000,18223.166666666668,9841.5,0.36446333333333336,0.5400543264525923,0,0,-10514,0
6,50000,39685.666666666664,1295.3333333333333,0.7937133333333333,0.032639827981555056,0,0,44376,0
7,500000,454099.1666666667,30126.5,0.9081983333333333,0.06634343819907179,0,0,-105979,0
8,100000,2247.6666666666665,798.5,0.022476666666666666,0.3552573038706807,0,0,11309,0
9,140000,10868.666666666666,1126.8333333333333,0.07763333333333333,0.10367723731828497,1,2,7566,0
10,20000,4486.5,2354.8333333333335,0.224325,0.5248709090233664,0,-1,-13912,0
